# Phase 3: Data Preparation

**CRISP-DM Phase Description:**  
This phase covers all activities to construct the final dataset from the initial raw data. Data preparation tasks are likely to be performed multiple times, and not in any prescribed order. This is typically the longest and most time-consuming phase of the CRISP-DM lifecycle.

---

In [1]:
# Standard library imports for this phase
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
%matplotlib inline

In [2]:
# Load the dataset from Phase 2 (update the path as needed)
DATA_PATH = 'path/to/your/data.csv'

import pandas as pd
import os
DATA_PATH = os.path.expanduser("~/Desktop/Data science project/data/processed/donordataset.csv")
df = pd.read_csv(DATA_PATH)

print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Loaded dataset: 109248 rows x 14 columns


,id,teacher_prefix,school_state,project_grade_category,project_subject_categories,project_subject_subcategories,teacher_number_of_previously_posted_projects,project_is_approved,price,quantity,cleaned_titles,cleaned_essays,cleaned_summary,isdigit_summary
0,p253737,mrs,in,grades_prek_2,literacy_language,esl_literacy,0,0,154.60,23,educational support english learners home,students english learners working english seco...,students_need_opportunities_practice_beginning...,0
1,p258326,mr,fl,grades_6_8,history_civics_health_sports,civics_government_teamsports,7,1,299.00,1,wanted projector hungry learners,students arrive school eager learn polite gene...,students_need_projector_help_viewing_education...,0
2,p182444,ms,az,grades_6_8,health_sports,health_wellness_teamsports,1,0,516.85,22,soccer equipment awesome middle school students,true champions not always ones win guts mia ha...,students_need_shine_guards_athletic_socks_socc...,0
3,p246581,mrs,ky,grades_prek_2,literacy_language_math_science,literacy_mathematics,4,1,232.90,4,techie kindergarteners,work unique school filled esl english second l...,students_need_engage_reading_math_way_inspire_...,0
4,p104768,mrs,tx,grades_prek_2,math_science,mathematics,1,1,67.98,4,interactive math tools,second grade classroom next year made around 2...,students_need_hands_practice_mathematics_fun_p...,0


---
### Task 1: Select Data

Decide on the data to be used for analysis. Consider which columns (features) and rows (records) to include or exclude based on:

- **Relevance:** Does this feature contribute to the data mining goal?
- **Data Quality:** Is the quality of this feature sufficient (e.g., too many missing values)?
- **Technical Constraints:** Are there limitations on data volume or specific feature types?

**Output:** A rationale for inclusion/exclusion of data, and the resulting subset.

**Instructions:** Select the columns and rows relevant to your analysis goal. Document your reasoning.

In [3]:
# TODO: Select the relevant columns and rows for your analysis.
# Document your rationale for including or excluding specific features.


selected_columns = [
    "teacher_prefix",
    "school_state",
    "project_grade_category",
    "project_subject_categories",
    "project_subject_subcategories",
    "teacher_number_of_previously_posted_projects",
    "price",
    "project_is_approved"
]

df_selected = df[selected_columns]

print(" SELECTED DATASET ")
print(f"Number of rows: {df_selected.shape[0]}")
print(f"Number of columns: {df_selected.shape[1]}")

display(df_selected.head())

 SELECTED DATASET 
Number of rows: 109248
Number of columns: 8


,teacher_prefix,school_state,project_grade_category,project_subject_categories,project_subject_subcategories,teacher_number_of_previously_posted_projects,price,project_is_approved
0,mrs,in,grades_prek_2,literacy_language,esl_literacy,0,154.60,0
1,mr,fl,grades_6_8,history_civics_health_sports,civics_government_teamsports,7,299.00,1
2,ms,az,grades_6_8,health_sports,health_wellness_teamsports,1,516.85,0
3,mrs,ky,grades_prek_2,literacy_language_math_science,literacy_mathematics,4,232.90,1
4,mrs,tx,grades_prek_2,math_science,mathematics,1,67.98,1


In [4]:
# Optional: Filter rows based on specific criteria
# Example: Remove rows where a critical field is missing or filter by a condition

df_selected = df_selected[
    df_selected['price'].notna()
]

print("  ROW FILTERING  ")
print(f"Shape after row selection: {df_selected.shape}")

  ROW FILTERING  
Shape after row selection: (109248, 8)


---
### Task 2: Clean Data

Raise data quality to the level required by the selected analysis techniques. Cleaning activities include:

- **Handle Missing Values:** Impute missing values (mean, median, mode, forward/backward fill) or remove rows/columns with excessive missing data.
- **Correct Errors:** Fix inaccurate or corrupted data entries.
- **Remove Duplicates:** Eliminate exact or near-duplicate records.
- **Handle Outliers:** Decide how to treat extreme values (keep, cap, transform, or remove).

**Instructions:** Apply appropriate cleaning techniques to address the data quality issues identified in Phase 2, Task 4.

In [5]:
# TODO: Handle missing values.
# Choose an appropriate strategy for each column.

import numpy as np

df_clean = df_selected.copy()

numerical_cols = df_clean.select_dtypes(include=np.number).columns
df_clean[numerical_cols] = df_clean[numerical_cols].fillna(df_clean[numerical_cols].median())

categorical_cols = df_clean.select_dtypes(include=['object', 'string']).columns
for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print(" MISSING VALUES AFTER CLEANING ")
print(f"Missing values remaining: {df_clean.isnull().sum().sum()}")

 MISSING VALUES AFTER CLEANING 
Missing values remaining: 0


In [6]:
# Remove duplicate records

before = len(df_clean)

df_clean = df_clean.drop_duplicates()

after = len(df_clean)

print("  DUPLICATE REMOVAL  ")
print(f"Removed {before - after} duplicate rows. Remaining: {after} rows.")

  DUPLICATE REMOVAL  
Removed 403 duplicate rows. Remaining: 108845 rows.


In [7]:
# Handle outliers using IQR capping (winsorizing)

def cap_outliers_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    dataframe[column] = dataframe[column].clip(lower=lower_bound, upper=upper_bound)
    return dataframe

#numerical columns except target
numerical_cols = [
    col for col in df_clean.select_dtypes(include='number').columns
    if col != 'project_is_approved'
]

print("=== OUTLIER HANDLING (IQR CAPPING) ===")
print("Columns processed:", numerical_cols)

# capping
for col in numerical_cols:
    df_clean = cap_outliers_iqr(df_clean, col)

print("\nOutliers handled using IQR capping (winsorizing).")

# Verify target variable unchanged
print("\nTarget distribution after outlier handling:")
print(df_clean['project_is_approved'].value_counts())

=== OUTLIER HANDLING (IQR CAPPING) ===
Columns processed: ['teacher_number_of_previously_posted_projects', 'price']

Outliers handled using IQR capping (winsorizing).

Target distribution after outlier handling:
project_is_approved
1    92307
0    16538
Name: count, dtype: int64


---
### Task 3: Construct Data (Feature Engineering)

This task involves creating new attributes (features) derived from existing ones that may be more useful for modelling. Common techniques include:

- **Derived Attributes:** Create new features from existing ones (e.g., extracting `year`, `month`, `day` from a datetime column; computing `total_spend = price * quantity`).
- **Binning / Discretisation:** Convert continuous variables into categorical bins (e.g., age groups).
- **Encoding Categorical Variables:** Convert categorical features into numerical representations (e.g., one-hot encoding, label encoding).
- **Scaling / Normalisation:** Scale numerical features to a common range (e.g., Min-Max scaling, Standardisation).

**Instructions:** Create new features or transform existing ones to improve model performance.

In [8]:
# TODO: Create derived attributes / new features.

df_clean['experience_level'] = df_clean['teacher_number_of_previously_posted_projects'].apply(
    lambda x: 'New' if x == 0 else ('Experienced' if x < 5 else 'Highly Experienced')
)

print("New feature 'experience_level' created.")
df_clean[['teacher_number_of_previously_posted_projects', 'experience_level']].head()

New feature 'experience_level' created.


,teacher_number_of_previously_posted_projects,experience_level
0,0.0,New
1,7.0,Highly Experienced
2,1.0,Experienced
3,4.0,Experienced
4,1.0,Experienced


In [9]:
# TODO: Encode categorical variables.

categorical_cols = df_clean.select_dtypes(include=['object', 'string']).columns

df_clean = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

print("Categorical variables encoded.")
print(f"New shape: {df_clean.shape}")

Categorical variables encoded.
New shape: (108845, 512)


In [10]:
# TODO: Scale / normalise numerical features if required.

from sklearn.preprocessing import StandardScaler

print(" FEATURE SCALING ")

scaler = StandardScaler()

# Select numerical columns except target
numerical_cols = [
    col for col in df_clean.select_dtypes(include='number').columns
    if col != 'project_is_approved'
]

print("Columns scaled:", numerical_cols)

# Apply scaling
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])

print("\nFeature scaling completed successfully.")

# Verify target unchanged
print("\nTarget distribution:")
print(df_clean['project_is_approved'].value_counts())

 FEATURE SCALING 
Columns scaled: ['teacher_number_of_previously_posted_projects', 'price']

Feature scaling completed successfully.

Target distribution:
project_is_approved
1    92307
0    16538
Name: count, dtype: int64


---
### Task 4: Integrate Data

If your project uses multiple data sources, this task involves merging or combining them into a single, unified dataset. Activities include:

- **Merging Tables:** Join datasets on common keys (e.g., using `pd.merge()`).
- **Appending Records:** Concatenate datasets with the same structure (e.g., using `pd.concat()`).
- **Aggregation:** Summarise data at a different level of granularity.

**Instructions:** If using multiple data sources, merge or concatenate them below. If your project uses a single dataset, document that here and proceed to the next task.

In [11]:
# TODO: Integrate data from multiple sources (if applicable).

df_integrated = df_clean.copy()

print("  DATA INTEGRATION  ")
print("This project uses a single dataset, so no merging or concatenation was required.")
print(f"Integrated dataset shape: {df_integrated.shape}")

display(df_integrated.head())

  DATA INTEGRATION  
This project uses a single dataset, so no merging or concatenation was required.
Integrated dataset shape: (108845, 512)


,teacher_number_of_previously_posted_projects,price,project_is_approved,teacher_prefix_mr,teacher_prefix_mrs,teacher_prefix_ms,teacher_prefix_teacher,school_state_al,school_state_ar,school_state_az,school_state_ca,school_state_co,school_state_ct,school_state_dc,school_state_de,school_state_fl,school_state_ga,school_state_hi,school_state_ia,school_state_id,school_state_il,school_state_in,school_state_ks,school_state_ky,school_state_la,school_state_ma,school_state_md,school_state_me,school_state_mi,school_state_mn,school_state_mo,school_state_ms,school_state_mt,school_state_nc,school_state_nd,school_state_ne,school_state_nh,school_state_nj,school_state_nm,school_state_nv,school_state_ny,school_state_oh,school_state_ok,school_state_or,school_state_pa,school_state_ri,school_state_sc,school_state_sd,school_state_tn,school_state_tx,school_state_ut,school_state_va,school_state_vt,school_state_wa,school_state_wi,school_state_wv,school_state_wy,project_grade_category_grades_6_8,project_grade_category_grades_9_12,project_grade_category_grades_prek_2,project_subject_categories_appliedlearning_health_sports,project_subject_categories_appliedlearning_history_civics,project_subject_categories_appliedlearning_literacy_language,project_subject_categories_appliedlearning_math_science,project_subject_categories_appliedlearning_music_arts,project_subject_categories_appliedlearning_specialneeds,project_subject_categories_appliedlearning_warmth_care_hunger,project_subject_categories_health_sports,project_subject_categories_health_sports_appliedlearning,project_subject_categories_health_sports_history_civics,project_subject_categories_health_sports_literacy_language,project_subject_categories_health_sports_math_science,project_subject_categories_health_sports_music_arts,project_subject_categories_health_sports_specialneeds,project_subject_categories_health_sports_warmth_care_hunger,project_subject_categories_history_civics,project_subject_categories_history_civics_appliedlearning,project_subject_categories_history_civics_health_sports,project_subject_categories_history_civics_literacy_language,project_subject_categories_history_civics_math_science,project_subject_categories_history_civics_music_arts,project_subject_categories_history_civics_specialneeds,project_subject_categories_history_civics_warmth_care_hunger,project_subject_categories_literacy_language,project_subject_categories_literacy_language_appliedlearning,project_subject_categories_literacy_language_health_sports,project_subject_categories_literacy_language_history_civics,project_subject_categories_literacy_language_math_science,project_subject_categories_literacy_language_music_arts,project_subject_categories_literacy_language_specialneeds,project_subject_categories_literacy_language_warmth_care_hunger,project_subject_categories_math_science,project_subject_categories_math_science_appliedlearning,project_subject_categories_math_science_health_sports,project_subject_categories_math_science_history_civics,project_subject_categories_math_science_literacy_language,project_subject_categories_math_science_music_arts,project_subject_categories_math_science_specialneeds,project_subject_categories_math_science_warmth_care_hunger,project_subject_categories_music_arts,project_subject_categories_music_arts_appliedlearning,project_subject_categories_music_arts_health_sports,project_subject_categories_music_arts_history_civics,project_subject_categories_music_arts_specialneeds,project_subject_categories_music_arts_warmth_care_hunger,project_subject_categories_specialneeds,project_subject_categories_specialneeds_health_sports,project_subject_categories_specialneeds_music_arts,project_subject_categories_specialneeds_warmth_care_hunger,project_subject_categories_warmth_care_hunger,project_subject_subcategories_appliedsciences_charactereducation,project_subject_subcategories_appliedsciences_civics_government,project_subject_subcategories_appliedsciences_college_careerprep,project_subject_subcategories_a

In [12]:
# Optional: Verify the integrated data
# df_integrated.head()

---
### Task 5: Format Data

This final preparation task ensures the data is in the correct format for the modelling tools. Activities include:

- **Data Type Conversions:** Ensure all columns have appropriate data types (e.g., numeric, datetime, categorical).
- **Column Reordering:** Arrange columns in a logical order (e.g., features first, target last).
- **Renaming:** Give columns clear, descriptive names.
- **Saving the Prepared Dataset:** Export the final, clean dataset for use in subsequent phases.

**Instructions:** Apply any final formatting changes and save the prepared dataset.

In [13]:
# TODO: Apply final formatting — data types, column order, renaming.

df_final = df_clean.copy()

# Ensure target is integer
df_final['project_is_approved'] = df_final['project_is_approved'].astype(int)

# Rename column for clarity
df_final = df_final.rename(columns={
    'teacher_number_of_previously_posted_projects': 'teacher_experience'
})

# Move target to the end
target_col = 'project_is_approved'
feature_cols = [col for col in df_final.columns if col != target_col]
df_final = df_final[feature_cols + [target_col]]

print("df_final created successfully.")
print(df_final.shape)

df_final created successfully.
(108845, 512)


In [14]:
print("df exists:", 'df' in globals())
print("df_selected exists:", 'df_selected' in globals())
print("df_clean exists:", 'df_clean' in globals())
print("df_final exists:", 'df_final' in globals())

df exists: True
df_selected exists: True
df_clean exists: True
df_final exists: True


In [15]:
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
TEST_SIZE = 0.2
TARGET_COL = 'project_is_approved'

X = df_final.drop(columns=[TARGET_COL])
y = df_final[TARGET_COL]

print("  TEST DESIGN  ")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target values:", y.unique())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

  TEST DESIGN  
X shape: (108845, 511)
y shape: (108845,)
Target values: [0 1]
Training set: (87076, 511)
Test set: (21769, 511)


In [16]:
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
TEST_SIZE = 0.2
TARGET_COL = 'project_is_approved'

X = df_final.drop(columns=[TARGET_COL])
y = df_final[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y
)

In [17]:
# TODO: Verify the final prepared dataset.

print("=" * 60)
print("FINAL PREPARED DATASET SUMMARY")
print("=" * 60)
print(f"Shape: {df_final.shape}")
print(f"Missing values: {df_final.isnull().sum().sum()}")
print(f"Duplicates: {df_final.duplicated().sum()}")
print("\nColumn types:")
print(df_final.dtypes)
df_final.head()

FINAL PREPARED DATASET SUMMARY
Shape: (108845, 512)
Missing values: 0
Duplicates: 538

Column types:
teacher_experience                                             float64
price                                                          float64
teacher_prefix_mr                                                 bool
teacher_prefix_mrs                                                bool
teacher_prefix_ms                                                 bool
                                                                ...   
project_subject_subcategories_visualarts_warmth_care_hunger       bool
project_subject_subcategories_warmth_care_hunger                  bool
experience_level_Highly Experienced                               bool
experience_level_New                                              bool
project_is_approved                                              int64
Length: 512, dtype: object


,teacher_experience,price,teacher_prefix_mr,teacher_prefix_mrs,teacher_prefix_ms,teacher_prefix_teacher,school_state_al,school_state_ar,school_state_az,school_state_ca,school_state_co,school_state_ct,school_state_dc,school_state_de,school_state_fl,school_state_ga,school_state_hi,school_state_ia,school_state_id,school_state_il,school_state_in,school_state_ks,school_state_ky,school_state_la,school_state_ma,school_state_md,school_state_me,school_state_mi,school_state_mn,school_state_mo,school_state_ms,school_state_mt,school_state_nc,school_state_nd,school_state_ne,school_state_nh,school_state_nj,school_state_nm,school_state_nv,school_state_ny,school_state_oh,school_state_ok,school_state_or,school_state_pa,school_state_ri,school_state_sc,school_state_sd,school_state_tn,school_state_tx,school_state_ut,school_state_va,school_state_vt,school_state_wa,school_state_wi,school_state_wv,school_state_wy,project_grade_category_grades_6_8,project_grade_category_grades_9_12,project_grade_category_grades_prek_2,project_subject_categories_appliedlearning_health_sports,project_subject_categories_appliedlearning_history_civics,project_subject_categories_appliedlearning_literacy_language,project_subject_categories_appliedlearning_math_science,project_subject_categories_appliedlearning_music_arts,project_subject_categories_appliedlearning_specialneeds,project_subject_categories_appliedlearning_warmth_care_hunger,project_subject_categories_health_sports,project_subject_categories_health_sports_appliedlearning,project_subject_categories_health_sports_history_civics,project_subject_categories_health_sports_literacy_language,project_subject_categories_health_sports_math_science,project_subject_categories_health_sports_music_arts,project_subject_categories_health_sports_specialneeds,project_subject_categories_health_sports_warmth_care_hunger,project_subject_categories_history_civics,project_subject_categories_history_civics_appliedlearning,project_subject_categories_history_civics_health_sports,project_subject_categories_history_civics_literacy_language,project_subject_categories_history_civics_math_science,project_subject_categories_history_civics_music_arts,project_subject_categories_history_civics_specialneeds,project_subject_categories_history_civics_warmth_care_hunger,project_subject_categories_literacy_language,project_subject_categories_literacy_language_appliedlearning,project_subject_categories_literacy_language_health_sports,project_subject_categories_literacy_language_history_civics,project_subject_categories_literacy_language_math_science,project_subject_categories_literacy_language_music_arts,project_subject_categories_literacy_language_specialneeds,project_subject_categories_literacy_language_warmth_care_hunger,project_subject_categories_math_science,project_subject_categories_math_science_appliedlearning,project_subject_categories_math_science_health_sports,project_subject_categories_math_science_history_civics,project_subject_categories_math_science_literacy_language,project_subject_categories_math_science_music_arts,project_subject_categories_math_science_specialneeds,project_subject_categories_math_science_warmth_care_hunger,project_subject_categories_music_arts,project_subject_categories_music_arts_appliedlearning,project_subject_categories_music_arts_health_sports,project_subject_categories_music_arts_history_civics,project_subject_categories_music_arts_specialneeds,project_subject_categories_music_arts_warmth_care_hunger,project_subject_categories_specialneeds,project_subject_categories_specialneeds_health_sports,project_subject_categories_specialneeds_music_arts,project_subject_categories_specialneeds_warmth_care_hunger,project_subject_categories_warmth_care_hunger,project_subject_subcategories_appliedsciences_charactereducation,project_subject_subcategories_appliedsciences_civics_government,project_subject_subcategories_appliedsciences_college_careerprep,project_subject_subcategories_appliedsciences_communityservice,project_subjec

In [18]:
# TODO: Save the prepared dataset for use in Phase 4 (Modelling).

import os
OUTPUT_PATH = os.path.expanduser("~/Desktop/prepared_donordataset.csv")
df_final.to_csv(OUTPUT_PATH, index=False)
print(f"Prepared dataset saved to: {OUTPUT_PATH}")

Prepared dataset saved to: /Users/ahmedsaleh/Desktop/prepared_donordataset.csv


In [19]:
import os
OUTPUT_PATH = os.path.expanduser("~/Desktop/prepared_donordataset.csv")

df_final.to_csv(OUTPUT_PATH, index=False)

print("=== DATA SAVED FOR PHASE 4 ===")
print(f"File saved at: {OUTPUT_PATH}")
print(f"Dataset shape: {df_final.shape}")

=== DATA SAVED FOR PHASE 4 ===
File saved at: /Users/ahmedsaleh/Desktop/prepared_donordataset.csv
Dataset shape: (108845, 512)
